# Spreadsheet to Plain Text Files

This notebook takes a spreadsheet and turns it into a set of plaintext files as commonly used for corpus linguistic tools like [AntConc](https://www.laurenceanthony.net/software/antconc/).

## Instructions 

1. Upload your spreadsheet into the folder panel on the left.
2. Run the first cell + follow the prompts to make decisions about how to create your output files.
3. Download and examine the resulting zip file.

In [ ]:
%pip install -q ipywidgets
import csv
import glob
import zipfile
import ipywidgets
from IPython.display import HTML

target_files = glob.glob("**.csv")

def extract_headers(csv_filepath):
    "Extract header rows from csv at the given location."
    with open(csv_filepath, 'r') as f:
        reader = csv.reader(f)
        return next(reader)

def update_column_selection(change):
    selected_file = change['new']
    headers = extract_headers(selected_file)
    column_selector.disabled=False
    column_selector.options=headers

def ready_to_generate(change):
    selected_columns = change['new']
    button.disabled=False

def generate_zip(button):
    csv_file = file_selector.value
    output_zip = output_name.value
    column = column_selector.value
    
    with open(file_selector.value, 'r') as f, zipfile.ZipFile(output_name.value, 'w') as z:
        reader = csv.DictReader(f)
        for i, row in enumerate(reader):
            z.writestr(f"{i}.txt", row[column])
            
    if not output_zip.endswith(".zip"):
        output_zip += ".zip"
        
    with output:
        display(HTML(f'<a href="{output_zip}" download="{output_zip}">Download your zip file</a>'))

file_selector = ipywidgets.Select(options=target_files, description="Choose file:")
column_selector = ipywidgets.Select(options=[], disabled=True, description="Choose columns:")
output_name = ipywidgets.Text("extracted.zip", description='Zip file name:')
button = ipywidgets.Button(description="Extract as text files", disabled=True)
button.on_click(generate_zip)
output = ipywidgets.Output()

file_selector.observe(update_column_selection, names=['value'])
column_selector.observe(ready_to_generate, names=['value'])

ipywidgets.GridBox([file_selector, column_selector, output_name, button, output])